# CS 687 · Homework 1 — Project 1, Checkpoint 1

**Tokenization, embeddings, and the information-theoretic view**

This notebook follows the lecture notes section by section. It is intended to run in Google Colab. You may instead run it on your own computer if you can set up the required Python and Jupyter environment yourself. Nothing in this homework requires a GPU.

The homework contains two substantial programming tasks, four short guided calculations, and four report responses. Every code answer cell is marked clearly and followed by an immediate check, so you can correct it before continuing. The four report cells are grouped near the end for easy review. The final check reruns the guided calculations and complete programming-test suite; always run it before submitting your notebook.

| Notebook section | Notes section |
|---|---|
| 1. Setup | — |
| 2. From next-token probability to loss | 1.2, 1.3 |
| 3. From text to byte tokens, then BPE training | 2.2 |
| 4. Consequences of tokenizer design | 2.3 |
| 5. The fertility experiment | 2.3 |
| 6. Task 2: training pairs by sliding window | 3.2 |
| 7. From token IDs to input embeddings | 3.1, 3.2 |
| 8. Units: nats, perplexity, bits per byte | 1.3 |
| 9. The embedding-parameter computation | 2.3 |
| 10. Final checks | — |
| 11. Report | synthesis |


## 1 · Setup

If you are using Colab, the cell below clones the starter repository from GitHub.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_NAME = "cs687-hw01-p1c1-test"
REPO_URL = "https://github.com/mk-er/cs687-hw01-p1c1-test.git"
RELEASE = "v0.18-test"

try:
    import google.colab  # type: ignore
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(
            ["git", "-c", "advice.detachedHead=false", "clone", "--branch", RELEASE,
             "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
else:
    repo_root = Path.cwd().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    if not (repo_root / "notebook_support.py").is_file():
        raise RuntimeError(
            "Could not find notebook_support.py. Open this notebook from "
            "somewhere inside the Homework 1 repository."
        )

sys.path.insert(0, str(repo_root))
from notebook_support import prepare_notebook
repo_root = prepare_notebook(repo_root)


## 2 · From next-token probability to loss — 4 points

At each sequence position, a language model produces a probability distribution over the vocabulary. Training evaluates the probability assigned to the token that actually occurred. If that probability is $p$, the loss contributed by the position is $-\ln p$. Natural logarithms make the unit **nats per token**.

A probability near one for the correct token gives a loss near zero. A probability near zero gives a large loss(unbounded), so confident mistakes are penalized strongly. Exponentiating an average loss in nats gives perplexity, the effective number of equally likely choices represented by that loss.

In the next cell, calculate the observed-token probability, its negative log-likelihood, and the corresponding perplexity from the supplied probability distribution.


In [ ]:
import torch

probabilities = torch.tensor([0.05, 0.15, 0.70, 0.10])
true_token_id = 2

# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute the probability, loss, and perplexity.")
# ================= YOUR CODE ENDS HERE ===================

print(f'probability of the observed token: {true_probability:.2f}')
print(f'negative log-likelihood       : {observed_token_loss_nats:.3f} nats')
print(f'perplexity                    : {token_perplexity:.3f}')


In [ ]:
from notebook_support import check_probability_calculation

check_probability_calculation(
    probabilities, true_token_id, true_probability,
    observed_token_loss_nats, token_perplexity
)


## 3 · From text to byte tokens

Before BPE learns anything, the tokenizer converts the input text to UTF-8 bytes. A byte is an integer from 0 through 255, so the initial vocabulary contains exactly 256 tokens: token ID `i` represents the single-byte value `bytes([i])`. For example, token ID `65` represents `bytes([65])`, which Python displays as `b'A'`. This mapping is fixed rather than learned.

ASCII characters, including the English alphabet, each use one UTF-8 byte and therefore begin as one token. Many characters outside ASCII use multiple bytes. For example, `ş` uses two UTF-8 bytes and is therefore initially represented by two tokens, while many emoji begin as four. Those separate byte tokens still preserve the complete text: concatenating the bytes and decoding them as UTF-8 recovers the original character.

The cell below shows the actual initial representation used by `BPETokenizer`.


In [ ]:
# Compare characters that occupy different numbers of UTF-8 bytes.
examples = ['A', 'a', '!', 'ş', 'ğ', '🧩']

print(f'{"Character":<12}{"UTF-8 bytes":<24}Initial token IDs')
print('-' * 58)
for character in examples:
    # Each byte value is also its initial token ID.
    encoded = character.encode('utf-8')
    token_ids = list(encoded)
    print(f'{character!r:<12}{encoded!r:<24}{token_ids}')


### Task 1 — Learn frequent byte sequences — 14 points

BPE shortens these initial representations by repeatedly replacing frequent adjacent token pairs with new token IDs. If the two bytes representing `ş` occur together frequently enough, for example, BPE may assign their pair one new ID, allowing later occurrences to be represented by a single token. The algorithm does not know that the pair forms a character; it merges the pair because it is frequent. The original byte tokens remain in the vocabulary as a fallback for any sequence that has not received a learned token.

Before implementing the training loop, open `cs687/tokenizer.py` and read `BPETokenizer.__init__`, `_pretokenize`, and `_apply_merge`. These define the two dictionaries you will update, the structure of `chunks`, and the helper used to apply each learned merge. You must not modify those methods.

The cell below imports the supplied `BPETokenizer` class and asks you to implement its missing `train` method.

The algorithm:

1. Pre-tokenize the text into chunks. This is done for you; `chunks` is a list of lists of byte values.
2. Repeat `num_merges` times:
   1. Count how often each pair of adjacent symbols occurs, across all chunks. Pairs never span two chunks.
   2. Stop early if there are no pairs left.
   3. Find the most frequent pair.
   4. Give it a new identifier, starting at 256, so that on step number `step` the new identifier is `256 + step`.
   5. Record the rule in `self.merges`, and record the byte string of the new symbol in `self.vocab`. The byte string of a merged symbol is the concatenation of the byte strings of its two parts.
   6. Rewrite every chunk, replacing each occurrence of the pair with the new identifier.

Two hints. `counts.update(zip(chunk, chunk[1:]))` counts the adjacent pairs of one chunk. `max(counts, key=counts.get)` returns the most frequent pair.

**Write your loop in the cell below, replacing the `raise NotImplementedError` line.**

The public check immediately afterward runs 18 tests. Four exercise supplied special-token methods after training; you do not need to implement those methods yourself.


In [ ]:
from collections import Counter
from cs687.tokenizer import BPETokenizer


def train(self, text: str, num_merges: int) -> None:
    chunks = self._pretokenize(text)

    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 1: implement BPETokenizer.train")
    # ================= YOUR CODE ENDS HERE ===================


In [ ]:
from notebook_support import run_task1_tests

run_task1_tests(train)


### Inspect what the tokenizer learned

The next cell trains on one short sentence and displays every learned merge. Notice that the original representation uses only byte-token IDs from 0 through 255, whereas every learned token receives a new ID beginning at 256. A learned token may stand for several bytes even though it occupies only one position in the final token sequence. Later merges can also build longer byte sequences from symbols created by earlier merges.


In [ ]:
# Use a deliberately small corpus and merge budget so every rule is visible.
tok = BPETokenizer()
tok.train('the model predicts the next token in the sequence ', 20)

print(f'learned {len(tok.merges)} merge rules:\n')
print(f'{"Learned byte sequence":<24}{"Current one-token representation":<36}Initial byte-token representation')
print('-' * 93)

# Display what each learned ID represents, down to the original byte IDs.
for new_id in sorted(tok.merges.values()):
    byte_sequence = tok.vocab[new_id]
    print(f'{byte_sequence!r:<24}{str([new_id]):<36}{list(byte_sequence)}')


**Pause and inspect the table.** Choose one learned token that represents at least three bytes. Identify its new token ID and the original byte-token IDs that it replaces. Then find an earlier learned token that could have participated in building it. This illustrates that later BPE merges can combine symbols created by earlier merges.


### Representing unseen text

The words below do not occur in the tokenizer-training sentence above. They are still representable because the initial vocabulary contains all 256 byte values. Learned merges can shorten a token sequence, but they are not required for the tokenizer to encode the text.


In [ ]:
# Encode strings absent from the tiny training sentence above.
for word in ['hugs', 'bug', 'öğrencilerimizden', 'tokens \U0001f9e9']:
    ids = tok.encode(word)
    pieces = [tok.vocab[i] for i in ids]
    print(f'{word!r:22} -> {len(ids):2d} tokens: {pieces}')
    assert tok.decode(ids) == word
print('\nEvery round trip is exact. The code is lossless.')


The longer representation of `öğrencilerimizden` is partly a consequence of UTF-8. ASCII characters use one byte each, but characters outside ASCII use multiple bytes. For example, `ö` is encoded by the two bytes `b'\xc3'` and `b'\xb6'`, while `ğ` is encoded by `b'\xc4'` and `b'\x9f'`. Therefore, the four pieces `b'\xc3', b'\xb6', b'\xc4', b'\x9f'` together represent `öğ`, not only `ö`.

This tiny tokenizer was trained on an English sentence, so it did not learn merges for those Turkish byte sequences. A tokenizer trained on text containing `ö` or `ğ` frequently could merge each recurring byte sequence into one token. Until then, the original byte tokens remain available, which is why the text is still represented exactly and decodes without loss.


### Special tokens represent trusted structure

BPE tokens are learned from frequent byte sequences in ordinary text. A **special token** serves a different purpose: it marks structure that the model should treat as meaningful, such as the boundary between two packed documents. It is reserved after BPE training, receives an ID above the learned merge IDs, and bypasses the merge process so that the complete marker is always represented by exactly one token.

The supplied `add_special_tokens` method is complete; you do not need to implement it. Run the demonstration below and compare the representation of `<|endoftext|>` before and after it is registered.


In [ ]:
boundary = '<|endoftext|>'
ordinary_ids = tok.encode(boundary)

special_tokens = tok.add_special_tokens((boundary,))
boundary_id = special_tokens[boundary]
packed_text = f'first document{boundary}second document'
packed_ids = tok.encode(packed_text)

print('before registration:', ordinary_ids)
print('reserved boundary ID:', boundary_id)
print('packed text IDs      :', packed_ids)
print('boundary occurrences :', packed_ids.count(boundary_id))

assert len(ordinary_ids) > 1
assert packed_ids.count(boundary_id) == 1
assert tok.decode(packed_ids) == packed_text


After registration, the boundary marker is one reserved ID rather than a sequence assembled from ordinary tokens. This gives the model an unambiguous structural signal. However, this teaching implementation also converts the literal string `<|endoftext|>` inside ordinary input into the special ID. A production tokenizer therefore requires the caller to explicitly allow special tokens; otherwise untrusted text could imitate, or *forge*, a document boundary.


## 4 · Consequences of tokenizer design

The following activities illustrate three consequences of tokenization: the training corpus and vocabulary budget determine which strings receive short representations, spaces can become part of a token, and character boundaries are not presented to the language model as explicit input units.

The tokenizer used in the short example above learned only 20 merges from one sentence. For a more meaningful comparison, the setup cell creates a **new** `BPETokenizer` named `yours` and trains it with 500 merges on the supplied parallel English–Turkish corpus. You can open and explore that corpus at `data/parallel_en_tr.tsv`. Each data row contains one English sentence and its Turkish translation, separated by a tab; introductory lines beginning with `>` are comments and are skipped.

The setup cell also loads GPT-2's pretrained tokenizer through `tiktoken`. It does not retrain GPT-2's tokenizer: GPT-2's 50,257-token vocabulary was learned previously from a much larger and different corpus.


In [ ]:
# GPT-2's tokenizer is downloaded the first time it is used, so this cell
# needs an internet connection.
import tiktoken

gpt2 = tiktoken.get_encoding('gpt2')

# Load the aligned course corpus while ignoring blank and comment lines.
def load_parallel(path='data/parallel_en_tr.tsv'):
    english, turkish = [], []
    for line in open(path, encoding='utf-8').read().splitlines():
        if not line.strip() or line.startswith('>'):
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            english.append(parts[0].strip())
            turkish.append(parts[1].strip())
    return english, turkish


# Train the course tokenizer used throughout the remaining comparisons.
english, turkish = load_parallel()
training_text = ' '.join(english) + ' ' + ' '.join(turkish)

yours = BPETokenizer()
yours.train(training_text, 500)
print(f'Trained `yours` with 500 merges on {len(english)} aligned sentence pairs.')


### Explore tokenizer segmentations

The next cell contains examples chosen to show different outcomes. GPT-2 uses fewer pieces for `tokenization` and `strawberry`, while the bilingual course tokenizer uses fewer for the Turkish corpus word `evlerinizden`. For each string, the cell shows both tokenizers' byte pieces and token counts and reports whether the string occurs in the course corpus.

The occurrence search treats uppercase and lowercase as the same, so `Technology` would count as an occurrence of `technology`; this applies only to the search, because the tokenizers themselves still distinguish capitalization. After running the supplied examples, add one string of your own. Consider how coverage in the small course corpus and the large difference in vocabulary size could explain the result.


In [ ]:
# Run the supplied contrasts, then add one string of your own.
examples = ['tokenization', 'strawberry', 'evlerinizden']

# Apply both tokenizers to exactly the same input strings.
for text in examples:
    your_ids = yours.encode(text)
    your_pieces = [yours.vocab[i] for i in your_ids]
    # Check whether the small course tokenizer encountered this string.
    occurrences = training_text.casefold().count(text.casefold())

    print(repr(text))
    gpt2_ids = gpt2.encode(text)
    gpt2_pieces = [gpt2.decode_single_token_bytes(i) for i in gpt2_ids]
    print(f'  GPT-2: {gpt2_pieces} (token count: {len(gpt2_ids)})')
    print(f'  yours: {your_pieces} (token count: {len(your_ids)})')
    print(f'  course-corpus occurrences (uppercase/lowercase treated as the same): {occurrences}\n')


### The leading-space effect

The pre-tokenizer attaches an ordinary space to the word that follows it. Consequently, `'the'` at the beginning of a chunk and `' the'` after a space begin as different byte sequences and can follow different merge paths. They may therefore produce different token sequences; when either complete form receives its own token ID, that ID also receives its own learned embedding.


In [ ]:
# Hold the letters fixed and change only the presence of the leading space.
for text in ['the', ' the']:
    gpt2_ids = gpt2.encode(text)
    your_ids = yours.encode(text)
    gpt2_pieces = [gpt2.decode_single_token_bytes(i) for i in gpt2_ids]
    your_pieces = [yours.vocab[i] for i in your_ids]
    print(f'{text!r:8} -> GPT-2 {gpt2_pieces}   yours {your_pieces}')

print('\nAdding the leading space changes the input bytes and can change the segmentation.')


### Why tokenization can make character-level questions difficult

A language model does not receive a word as a list of characters. The tokenizer first converts the text into token IDs, and the model receives learned vectors for those IDs. The cell below shows the token pieces used for `strawberry`. With GPT-2's tokenizer, for example, the word is divided into pieces such as `st`, `raw`, and `berry`, rather than into the individual letters `s`, `t`, `r`, and so on.

This process is lossless: decoding the token IDs still reconstructs `strawberry` exactly, so no characters have been discarded. However, the boundaries between individual characters are not supplied to the model as explicit input units. The model may learn spelling patterns indirectly from its training data, but answering a question such as "How many times does `r` occur in *strawberry*?" can therefore be less direct than it would be for a system that received one character at a time.


In [ ]:
# Inspect the model-facing pieces, then verify that decoding is lossless.
ids = gpt2.encode('strawberry')
pieces = [gpt2.decode_single_token_bytes(i) for i in ids]
print('Token pieces produced by GPT-2:', pieces)
print('Decoding all token IDs:', gpt2.decode(ids))


## 5 · The fertility experiment — 4 points

A tokenizer's **fertility** on a collection of text is the average number of tokens it produces per whitespace-separated word. Higher fertility means that the tokenizer fragments the language into more pieces. The experiment below measures the 101 aligned English–Turkish sentence pairs, so the two sides express approximately the same content in different languages. It also reports **English tokens per byte** and **Turkish tokens per byte**: the number of tokens produced divided by the number of UTF-8 bytes in the corresponding text. This is the normalization needed later to convert bits per token into bits per byte.

The first two rows form the controlled comparison. Both use this homework's BPE algorithm, the same 500-merge budget, and the same parallel corpus; only the text used to learn the merges changes. The first tokenizer learns from the English side alone, while the second learns from both languages. GPT-2 is included as an external reference, not as a controlled comparison, because it has a much larger vocabulary learned from a different corpus.


Before running the experiment, make a prediction: which of the two course tokenizers should have lower fertility on English, and which should have lower fertility on Turkish? Why? First complete the two normalization formulas in `measure`, and then compare your prediction with the table.


In [ ]:
# Measure fragmentation by both words and UTF-8 bytes.
def measure(tok, sentences):
    text = ' '.join(sentences)
    n_tokens = len(tok.encode(text))

    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Compute fertility and tokens per UTF-8 byte.")
# ================= YOUR CODE ENDS HERE ===================

    return {
        'fertility': fertility,
        'tokens_per_byte': tokens_per_byte,
    }


In [ ]:
from notebook_support import check_fertility_measure

check_fertility_measure(measure)


In [ ]:
# Controlled comparison: same algorithm and budget, different training text.
en_only = BPETokenizer()
en_only.train(' '.join(english), 500)

both = BPETokenizer()
both.train(' '.join(english) + ' ' + ' '.join(turkish), 500)

# GPT-2 is an external reference rather than part of the controlled pair.
rows = [
    ('course, English only', en_only),
    ('course, both languages', both),
    ('GPT-2', gpt2),
]

header = (
    f"{'tokenizer':24} {'EN fertility':>12} {'TR fertility':>12} {'TR/EN':>7} "
    f"{'EN tokens/byte':>14} {'TR tokens/byte':>14}"
)
print(header)
print('-' * len(header))
for name, tok in rows:
    en = measure(tok, english)
    tr = measure(tok, turkish)
    ratio = tr['fertility'] / en['fertility']
    print(
        f"{name:24} {en['fertility']:12.2f} {tr['fertility']:12.2f} "
        f"{ratio:7.2f} {en['tokens_per_byte']:14.3f} {tr['tokens_per_byte']:14.3f}"
    )


### Interpreting the fertility measurements

Compare the first two rows. The algorithm and merge budget are identical; only the tokenizer-training data differed. The resulting imbalance comes from which byte patterns that corpus made frequent enough to merge. A fixed token context holds fewer words when fertility is higher, and processing the same amount of text requires more token positions. Token-metered services may therefore charge more for that input, although the exact monetary and computational cost depends on the deployed system.

Before continuing, make sure you can explain why adding Turkish to the tokenizer-training corpus changes the Turkish measurements, and identify which tokens-per-byte value would be needed to convert a Turkish loss from bits per token to bits per byte.

This is tokenization as an **inductive bias**. Before the language model learns from its loss, the tokenizer has already decided which frequent strings receive short sequences and their own token embeddings. Other text remains representable because the tokenizer can fall back to bytes, but the model must process it through more token positions.


## 6 · Task 2 — training pairs by sliding window — 10 points

An autoregressive language model learns to predict the next token at every position. We therefore turn one long token stream into many fixed-length input–target pairs. For example:

```text
ids = [10, 20, 30, 40, 50, 60]
context_len = 3
stride = 2

start 0:  input = [10, 20, 30]   target = [20, 30, 40]
start 2:  input = [30, 40, 50]   target = [40, 50, 60]
```

Each target is the corresponding input shifted forward by one position. A window of `context_len` input tokens therefore requires one additional token for its final target. The `stride` determines how far the next window moves and how much consecutive windows overlap. Before continuing, confirm that the example produces two pairs, that `2` is its final valid starting position, and that token `30` occurs in both input windows.

In general, for a window beginning at position `s`:

```
input  = ids[s     : s + T]        that is, (y_1, ..., y_T)
target = ids[s + 1 : s + T + 1]    that is, (y_2, ..., y_{T+1})
```

One forward pass can then produce a next-token loss at every position in the window.

Walk a window across `ids` in steps of `stride`, appending one tensor to each list per window. Stop at `len(ids) - context_len`, because the final window needs one extra token for its target.

**Write your loop in the cell below.**


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class NextTokenDataset(Dataset):
    def __init__(self, ids, context_len, stride):
        self.inputs, self.targets = [], []

        # ================= YOUR CODE STARTS HERE =================
        raise NotImplementedError("Task 2: fill self.inputs and self.targets")
        # ================= YOUR CODE ENDS HERE ===================

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


print("class defined")


In [ ]:
from notebook_support import run_task2_tests

run_task2_tests(NextTokenDataset)


### Inspecting consecutive input–target pairs

The first two windows make both relationships visible: every target is shifted by one token, and the windows overlap because the stride is smaller than the context length.


In [ ]:
# Use `yours`, the 500-merge course tokenizer trained on both languages.
tokenizer = yours

# Build overlapping input-target windows from a short token sequence.
ids = tokenizer.encode(' '.join(english[:12]))
ds = NextTokenDataset(ids, context_len=10, stride=5)

# Decode two consecutive pairs so the shift and overlap are visible.
for pair_index in range(2):
    x, y = ds[pair_index]
    print(f'window {pair_index + 1} (start {pair_index * 5})')
    print('  input :', [tokenizer.decode([i]) for i in x.tolist()])
    print('  target:', [tokenizer.decode([i]) for i in y.tolist()])
    print('  input ids :', x.tolist())
    print('  target ids:', y.tolist())
    print()

print('Each target is shifted by one position; the two inputs overlap by five tokens.')


### Stride and overlap

A smaller stride produces more windows and reuses more of the same tokens. This provides more training pairs, but adjacent examples become more strongly correlated and require more computation. With `context_len=10` and `stride=5`, consecutive input windows overlap by five tokens.

Before running the next cell, order strides `10`, `5`, `2`, and `1` from the fewest to the most training pairs. Then check whether the measured counts match your prediction.


In [ ]:
# Hold the context length fixed while changing the distance between windows.
for stride in [10, 5, 2, 1]:
    n = len(NextTokenDataset(ids, context_len=10, stride=stride))
    print(f'stride {stride:2d} -> {n:4d} training pairs')


## 7 · From token IDs to input embeddings

`InputEmbedding` is a course-supplied PyTorch module rather than a built-in PyTorch class. Before running the next cell, open `cs687/embedding.py` and examine its definition. Identify the token-embedding table, the learned position-embedding table, and the line in `forward` that adds their outputs together.

The `InputEmbedding` layer created in this notebook is newly initialized, so its vectors do not yet encode meaningful relationships. During language-model training, the loss updates these vectors. Tokens used in related contexts may develop related representations, although the resulting geometry can reflect spelling, grammar, frequency, and other patterns—not only semantic meaning. The token embedding itself is context-independent; representations that depend on the surrounding sentence develop later as the Transformer processes the sequence.

`NextTokenDataset` returns one input–target pair at a time. PyTorch's `DataLoader` collects several pairs into a batch: `B` is the number of examples in the batch and `T` is the context length. The next cell first checks this batching step and then passes the input IDs through `InputEmbedding`. Token IDs of shape `(B, T)` go in, and floating-point representations of shape `(B, T, d_model)` come out.


In [ ]:
from cs687.embedding import InputEmbedding

# Compare one dataset item with a batch assembled by DataLoader.
single_x, single_y = ds[0]
loader = DataLoader(ds, batch_size=4, shuffle=False, drop_last=True)
batch_x, batch_y = next(iter(loader))

print('one input example :', tuple(single_x.shape))
print('batched inputs    :', tuple(batch_x.shape))
print('batched targets   :', tuple(batch_y.shape))
assert torch.equal(batch_x[:, 1:], batch_y[:, :-1])
print('Every target in the batch is shifted by one position.\n')

# Replace every token ID with a d_model-dimensional input representation.
emb = InputEmbedding(vocab_size=yours.vocab_size, d_model=64, max_len=128)
out = emb(batch_x)

print('token ID batch     :', tuple(batch_x.shape))
print('embedding output  :', tuple(out.shape))
assert out.shape == (4, 10, 64)
print('\nShape check passed.')


### Separating token and position vectors — 4 points

The input `batch_x` has shape `(B, T)`: it contains `T` token IDs for each of the `B` sequences. Looking up those IDs in the token-embedding table produces `token_vectors` with shape `(B, T, d_model)`. Separately, looking up the position numbers `0, 1, ..., T-1` produces `position_vectors` with shape `(T, d_model)`.

Every sequence uses the same vector for position 0, the same vector for position 1, and so on. PyTorch therefore reuses the `(T, d_model)` position table across the `B` sequences when the tensors are added. For batch item `b` and position `t`, the calculation is

```text
combined[b, t, :] = token_vectors[b, t, :] + position_vectors[t, :]
```

In the next cell, complete these three steps inside the `torch.no_grad()` block:

1. Pass `batch_x` to the token-embedding table `emb.tok` to obtain `token_vectors`.
2. Pass `positions` to the position-embedding table `emb.pos` to obtain `position_vectors`.
3. Add `token_vectors` and `position_vectors` to obtain `combined`.

Inspect `cs687/embedding.py` to connect these steps to `InputEmbedding.forward`. Dropout is disabled so that your manually computed sum can be compared exactly with the module's output.


In [ ]:
# Expose the two learned components that InputEmbedding adds together.
emb.eval()
positions = torch.arange(batch_x.shape[1])

with torch.no_grad():
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Compute the token, position, and combined vectors.")
# ================= YOUR CODE ENDS HERE ===================
    module_output = emb(batch_x)

print('token vectors   :', tuple(token_vectors.shape))
print('position vectors:', tuple(position_vectors.shape))
print('combined output :', tuple(combined.shape))


In [ ]:
from notebook_support import check_embedding_components

check_embedding_components(
    token_vectors, position_vectors, combined, module_output
)


### Embedding lookup as one-hot matrix multiplication

Section 3.1 writes the embedding matrix as $\boldsymbol\Omega_e \in \mathbb{R}^{d_{model} \times V}$, so multiplying it by a one-hot vector selects one column. PyTorch stores the transpose, with one row per token; `nn.Embedding` directly selects that row instead of performing the full multiplication.


In [ ]:
# Compare direct embedding lookup with multiplication by a one-hot vector.
emb.eval()
with torch.no_grad():
    direct = emb.tok(torch.tensor([3]))
    one_hot = torch.zeros(1, yours.vocab_size)
    one_hot[0, 3] = 1.0
    via_matmul = one_hot @ emb.tok.weight

print('largest difference:', float((direct - via_matmul).abs().max()))
print('identical to within floating-point error')


### Adding learned position information

Bare self-attention is permutation equivariant: reordering its input vectors merely reorders the corresponding outputs without otherwise changing them. Since word order changes meaning, the representation supplied to attention must explicitly combine what the token is with where it occurs. The learned position embedding below provides that information, so the same token at two positions receives different input representations.


In [ ]:
# Keep the token ID fixed and change only its position.
with torch.no_grad():
    same_token_twice = emb(torch.tensor([[5, 5]]))

difference = float((same_token_twice[0, 0] - same_token_twice[0, 1]).abs().max())
print('difference between the two positions:', round(difference, 4))
print('non-zero, because the position embedding differs')


## 8 · Units: nats, perplexity, and bits per byte — 4 points

Section 1.3 describes several equivalent ways to report predictive loss. If the average loss uses natural logarithms, it is measured in nats per token. Dividing by $\ln 2$ converts nats per token to bits per token:

$$
\frac{\text{nats}}{\text{token}} \div \ln 2 = \frac{\text{bits}}{\text{token}}.
$$

Multiplying by the tokenizer's measured tokens per byte then removes the dependence on how that tokenizer divides the text:

$$
\frac{\text{bits}}{\text{token}} \times \frac{\text{tokens}}{\text{byte}} = \frac{\text{bits}}{\text{byte}}.
$$

Complete the three conversion expressions below to reproduce the worked example from the notes. Keep track of the units at every step: tokens per byte, bits per token, and finally bits per byte.


In [ ]:
from cs687.units import perplexity, bits_per_token, bits_per_byte, compression_ratio

# Express the same average loss using several equivalent units.
loss_nats = 3.2
tokens_per_word, bytes_per_word = 1.3, 5.9

# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Convert the loss from nats/token to bits/byte.")
# ================= YOUR CODE ENDS HERE ===================

print(f'loss             = {loss_nats:.2f} nats/token')
print(f'perplexity       = exp({loss_nats:.2f}) = {perplexity(loss_nats):.1f}')
print(f'bits per token   = {loss_nats:.2f} / ln(2) = {loss_bits:.2f}')
print(f'tokens per byte  = {tokens_per_word:.1f} / {bytes_per_word:.1f} = {tokens_per_byte:.3f}')
print(f'bits per byte    = {loss_bits:.2f} × {tokens_per_byte:.3f} = {loss_bits_per_byte:.2f}')
print(f'compression size = {100 * compression_ratio(loss_nats, tokens_per_byte):.0f}% of the original')


In [ ]:
from notebook_support import check_unit_conversion

check_unit_conversion(
    loss_nats, tokens_per_word, bytes_per_word,
    tokens_per_byte, loss_bits, loss_bits_per_byte
)


### Using measured tokens per byte

Homework 1 does not train a language model, so the next cell combines a **hypothetical** loss of 4.1 nats per token with the bilingual course tokenizer's actual tokens-per-byte measurement on the English corpus. Before running it, predict what would happen to bits per byte if the loss stayed fixed but the tokenizer required more tokens per byte.


In [ ]:
# Combine a hypothetical model loss with a measured tokenizer statistic.
hypothetical_loss = 4.1
text = ' '.join(english)
measured_tokens_per_byte = len(yours.encode(text)) / len(text.encode('utf-8'))
hypothetical_bits_per_token = bits_per_token(hypothetical_loss)
hypothetical_bits_per_byte = (
    hypothetical_bits_per_token * measured_tokens_per_byte
)

print(f'hypothetical loss       : {hypothetical_loss:.2f} nats/token')
print(f'measured tokens per byte: {measured_tokens_per_byte:.3f}')
print(f'bits per token          : {hypothetical_bits_per_token:.2f}')
print(
    f'bits per byte           : {hypothetical_bits_per_token:.2f} × '
    f'{measured_tokens_per_byte:.3f} = {hypothetical_bits_per_byte:.2f}'
)


## 9 · Counting embedding-matrix parameters

A token-embedding table stores one vector of length `d_model` for every vocabulary item, so it contains $V \times d_{model}$ learned parameters. Before running the next cell, estimate the parameter counts for the small course embedding and GPT-2's token embedding. Notice that increasing a BPE vocabulary by 500 tokens adds $500 \times d_{model}$ parameters to this table, even if the rest of the model is unchanged.

GPT-2 shares, or *ties*, its input token-embedding weights with the output projection used to predict tokens. The final line shows how much larger the model would be if a separate output matrix of the same size were required.


In [ ]:
# Compare the notebook's small token table with GPT-2's token table.
course_V, course_d_model = yours.vocab_size, 64
gpt2_V, gpt2_d_model, gpt2_total = 50_257, 768, 124_000_000

course_embedding_params = course_V * course_d_model
gpt2_embedding_params = gpt2_V * gpt2_d_model

print(f'course: {course_V:,} × {course_d_model} = {course_embedding_params:,} parameters')
print(f'GPT-2 : {gpt2_V:,} × {gpt2_d_model} = {gpt2_embedding_params:,} parameters')
print(
    f'GPT-2 token embeddings are '
    f'{100 * gpt2_embedding_params / gpt2_total:.0f}% of its {gpt2_total:,} parameters.'
)

extra_tokens = 500
print(
    f'Adding {extra_tokens} course-vocabulary tokens would add '
    f'{extra_tokens * course_d_model:,} token-embedding parameters.'
)

untied_total = gpt2_total + gpt2_embedding_params
print(
    'Without weight tying, GPT-2 input and output token matrices would occupy '
    f'{100 * 2 * gpt2_embedding_params / untied_total:.0f}% of the enlarged model.'
)


## 10 · Run the public tests

The immediate checks above provide feedback after each guided calculation and programming task. This final cell repeats all four calculation checks and runs all 35 public programming tests against the two larger implementations currently loaded in the notebook. Always run it before submitting your notebook. It does not require you to copy anything into the repository's `.py` files. Staff grading will use a separate, staff-controlled test suite.


In [ ]:
from notebook_support import (
    check_embedding_components,
    check_fertility_measure,
    check_probability_calculation,
    check_unit_conversion,
    run_all_public_tests,
)

check_probability_calculation(
    probabilities, true_token_id, true_probability,
    observed_token_loss_nats, token_perplexity
)
check_fertility_measure(measure)
check_embedding_components(
    token_vectors, position_vectors, combined, module_output
)
check_unit_conversion(
    loss_nats, tokens_per_word, bytes_per_word,
    tokens_per_byte, loss_bits, loss_bits_per_byte
)

run_all_public_tests(train, NextTokenDataset)


## 11 · Report

**Name:**  
**Student ID:**   

Use the four cells below to explain what the notebook experiments show. Do not paste entire output tables or repeat the code line by line. Support your explanations with relevant values from your own run, and show the arithmetic where requested. These report cells are part of the notebook submission.


### 1 · From text to BPE tokens — 14 points

Use one concrete example from the notebook, such as `ş` or one of the learned multi-byte tokens, to trace the path from Unicode text to UTF-8 bytes, initial byte-token IDs, and any learned BPE token IDs. In the same explanation, answer both questions:

- Why can this tokenizer represent text that never appeared in its training corpus?
- Why must encoding replay the learned merge rules instead of learning new merges from each input?

Write approximately 100–150 words.

**Response:**

_Replace this line with your response._


### 2 · Training data as an inductive bias — 14 points

Use the first two rows of the fertility experiment as a controlled comparison. Report the observed English and Turkish fertility values, and explain why adding Turkish to the tokenizer-training corpus changes the result. Connect the result to one practical consequence of higher fertility, such as effective context length or the number of token positions processed.

Finally, explain briefly why the GPT-2 row is useful as a reference but is not part of the same controlled comparison. Write approximately 100–150 words.

**Response:**

_Replace this line with your response._


### 3 · From token IDs to Transformer inputs — 16 points

Explain the complete path from one token-ID sequence to the vectors supplied to a Transformer. Include:

- how `NextTokenDataset` constructs an input window and its one-position-shifted target;
- how `context_len`, `stride`, and `batch_size` affect the resulting tensors;
- the shapes before and after `InputEmbedding`; and
- why token vectors alone do not provide the order information required by a language model, and how position vectors address this problem.

Use one concrete shape or window from your notebook output. Write approximately 150–200 words.

**Response:**

_Replace this line with your response._


### 4 · Comparing models and tokenizer designs — 16 points

Assume a language model using the bilingual course tokenizer has an average loss of **4.1 nats per token** on the English side of the supplied corpus.

1. Use the measured English tokens-per-byte value from your notebook to convert this loss to bits per token and then bits per byte. Show the formula, substitution, and result.
2. Explain why bits per byte supports a fairer comparison between models with different tokenizers than nats per token.
3. The notebook uses `d_model = 64`. Calculate how many token-embedding parameters would be added by 500 additional vocabulary items. State one possible benefit and one possible cost of increasing the BPE vocabulary.

Show the calculation, then write approximately 100–150 words of explanation. Displayed calculations do not count toward this guideline.

**Response:**

_Replace this line with your response._


## 12 · Finishing the homework

Submit only this notebook. Before uploading it to Moodle, rename it to `homework01_colab_STUDENTNUMBER.ipynb`, replacing `STUDENTNUMBER` with your student number. For example: `homework01_colab_22206543.ipynb`.

Before submitting, restart the Colab runtime and run the notebook from top to bottom. Confirm that the final check reports that all four guided calculations and all 35 public programming tests passed, and make sure none of the four report cells still contains its placeholder. Rename and download the completed notebook, then use the final cell below to upload and check that exact file. Fix every reported error before uploading the same file to Moodle. The Moodle upload procedure will be announced before release.

The report explanations are the part that matters most. You may write the code with the help of a coding agent, but the quiz and the examination ask related questions without assistance, so make sure you can explain whatever you submit. Saved notebook output is not used as proof of correctness: the submitted notebook will be executed again from a clean environment.


### Check the saved submission file

In Colab, run the cell below and select the completed `.ipynb` file that you just downloaded. The checker examines the actual file that you will submit, including its required cells, tags, and answer regions. If it reports `READY`, submit that same file to Moodle without editing it again.

When working locally, run the equivalent terminal command shown by the cell instead.


In [ ]:
from notebook_support import check_saved_submission

check_saved_submission()
